<a href="https://colab.research.google.com/github/JoelvdR/medicalAI/blob/main/Group15_%5BMAM11A_2026%5D_V%26L_Practicum_Uncertainty_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Practicum Vision & Language NLP - Uncertainty Quantification with Multimodal LLMs

In this notebook, we will evaluate several multimodal Vision & Language LLMs on a Medical Visual Question Answering task. We will use the following dataset: [GMAI-MMBench: A Comprehensive Multimodal Evaluation Benchmark Towards General Medical AI.](https://huggingface.co/datasets/OpenGVLab/GMAI-MMBench) The dataset is released under the Apache 2.0 license, and you will work on a pre-processed subset prepared by us.

We will evaluate several models:
- `Qwen2-VL-2B-Instruct` and `Medgemma-4b-it` (smaller models that can be run directly on Colab; output files are also provided in case of issues).
- `Qwen2.5-VL-32B-Instruct` and `GPT-5-mini` (larger models - we provide the output files).

Output files are named according to the following template:
`outputs_{model_name}-sample_{sample_answers}-temp_{temperature}.tsv`

### Setup

- In Runtime, select Change runtime type, then choose **T4 GPU** as the hardware accelerator.  

- Log into Hugging Face and accept the license agreement for [Qwen2-VL-2B-Instruct](https://huggingface.co/Qwen/Qwen2-VL-2B-Instruct) and [Medgemma-4b-it](https://huggingface.co/google/medgemma-4b-it). Attention, this may take a while to be processed.

- Set your Hugging Face token as the variable **HF_TOKEN** in the Secrets tab (left menu, key icon) or paste your token in the "your_token" in the required field. Make sure not to accidentaly share it.


### Important Note

This challenging assignment gives you plenty of freedom to explore and be creative. You are expected to demonstrate independence, formulate your own hypotheses, and test them empirically. Since this is a cutting-edge research area with many open questions, take the opportunity to investigate, experiment, and follow your curiosity! :)


####Students: Name:
Emmanuella Ekpenyong - UvA id:16411773  
Name: Britt  Uiterdijk -	UvA id:13451308  
Name: Joël van der Roest- UvA id:14371839

In [1]:
from typing import List, Dict, Any, Tuple, Generator, Optional, Union
import random
import string, os, sys, glob
import pandas as pd
import numpy as np
from google.colab import userdata

from PIL import Image
from tqdm import tqdm
import torch
import ast

In [2]:
try:
  hf_token = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
  hf_token = "your_token"

os.environ["HF_TOKEN"] = hf_token

In [3]:
!gdown --folder https://drive.google.com/drive/folders/1dNoFV9GvDhMdTjM2P2oHW00-f9oPMxnf -O /content/data # download pre-processed dataset files + model responses
!unzip -q data/filtered_images.zip -d data/images/ # unzip the images folder

Retrieving folder contents
Retrieving folder 1MxSEpKZWI26Gl2dVdvRd2uMn7Eimr9dC model_responses
Processing file 1tG8zYy11gXQ-puRQocji6OOKogHR12Pi outputs_gpt5-mini_no-logprobs.tsv
Processing file 1GVFn2qKdDJxtEbbrufC0W7eULvAZMyPa outputs_medgemma-4b-it-temp_0.2.tsv
Processing file 1BAx2jUeuUbgRX6joWR94vdS0nsqCNf-I outputs_medgemma-4b-it-temp_0.6.tsv
Processing file 1JK7ACAkPVKIyEZ39RtVru3GDgUvMk_8J outputs_medgemma-4b-it-temp_1.0.tsv
Processing file 18cSUZl93yG167twxUY_gmEaMYXDOfi3q outputs_Qwen2-VL-2B-Instruct-temp_0.2.tsv
Processing file 1pxUlsk9rK67Uo9mOSxKKOebMzGxPYXS9 outputs_Qwen2-VL-2B-Instruct-temp_0.6.tsv
Processing file 1WlgSroLtv5STSufSC4qRM419KW5YRBod outputs_Qwen2-VL-2B-Instruct-temp_1.0.tsv
Processing file 1HjbP_JjNq1gkqsz-RiXff-c5M_tvPvb8 outputs_Qwen2.5-VL-32B-Instruct-temp_0.2.tsv
Processing file 11XjxhRI7j-xOEtnTYnE7-v8pFTOk6jIh outputs_Qwen2.5-VL-32B-Instruct-temp_0.6.tsv
Processing file 1FSfXTQj2ytMsCfO0Iy_xoeP0P-IqI-K7 outputs_Qwen2.5-VL-32B-Instruct-temp_1.0.tsv
Pr

# 1. Inference (run on GPUs)

## Load and Prepare Model Inputs

In [4]:
def add_options_dict(df_raw: pd.DataFrame) -> pd.DataFrame:  # prepare the multiple-choice question format
    option_cols = [col for col in df_raw.columns if col in list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")]

    def _make_dict(row):
        return {col: row[col] for col in option_cols if pd.notna(row[col])}

    out = df_raw.copy()
    out["options"] = out.apply(_make_dict, axis=1)
    return out

def construct_prompt(row) -> str:  # prepare the prompt for the model
    PROMPT_TEMPLATE = """Question: ({question})
Provide a concise answer and start your response with the letter of the selected option in square brackets. Options:
(
{options}
)
Your Answer:
"""
    question = row["question"]
    options = {
        letter: row[letter]
        for letter in string.ascii_uppercase[:5]
        if letter in row and row[letter] and str(row[letter]).strip() and not pd.isna(row[letter])
    }

    formatted_options = "\n".join(
        f"[{letter}] {option}"
        for letter, option in options.items()
    )

    return PROMPT_TEMPLATE.format(
        question=question,
        options=formatted_options
    )

In [ ]:
input_tsv = "data/GMAI_mm_bench_dataset.tsv"  # the pre-processed dataset we provide
df = pd.read_csv(input_tsv, sep="\t")

df = add_options_dict(df)
df["prompt"] = df.apply(lambda row: construct_prompt(row), axis=1)
data_records: List[dict] = df.to_dict(orient="records")

In [ ]:
df.head()  # inspect the data

,index,question,A,B,C,D,E,answer,category,clinical VQA task,department,perceptual granularity,modality,original task,image_path,options,prompt
0,398,Which of the following options best matches th...,fascia uncovered kidney parenchyma,lung,fascia covered kidney parenchyma,heart,clavicles,D,heart,Organ Recognition - Thorax,Cardiovascular Surgery,Contour level,X-ray,SEG_2D,398.jpg,"{'A': 'fascia uncovered kidney parenchyma', 'B...",Question: (Which of the following options best...
1,4322,"Observe the image, please select the option th...",grade 2 (minimal),grade 3 (moderate),grade 4 (severe),grade 1 (doubtful),NaN,A,grade 2 (minimal),Severity Grading,Sports Medicine,Image level,X-ray,CLS_2D,4322.jpg,"{'A': 'grade 2 (minimal)', 'B': 'grade 3 (mode...","Question: (Observe the image, please select th..."
2,2627,Considering the symptoms visible in the pictur...,COVID 19,viral pneumonia,normal chest x ray,NaN,NaN,B,viral pneumonia,Disease Diagnosis,Pulmonary Medicine,Image level,X-ray,CLS_2D,2627.jpg,"{'A': 'COVID 19', 'B': 'viral pneumonia', 'C':...",Question: (Considering the symptoms visible in...
3,70,Focus on the square-highlighted area of this X...,pulmonary mass,diffuse pulmonary nodule,pleural effusion,pulmonary calcification,NaN,A,pulmonary mass,Disease Diagnosis,Pulmonary Medicine,Box level,X-ray,DET_2D,70.jpg,"{'A': 'pulmonary mass', 'B': 'diffuse pulmonar...",Question: (Focus on the square-highlighted are...
4,4525,Choose the description that best matches the i...,negative for pneumonia,typical appearance of COVID-19,indeterminate appearance of COVID-19,atypical appearance of COVID-19,NaN,C,indeterminate appearance of COVID-19,Severity Grading,Pulmonary Medicine,Image level,X-ray,CLS_2D,4525.jpg,"{'A': 'negative for pneumonia', 'B': 'typical ...",Question: (Choose the description that best ma...


In [ ]:
len(df) # we have 300 image-question pairs

300

In [ ]:
# Q1
for column in ["clinical VQA task", "department"]:
    counts = df[column].fillna("Missing").value_counts()
    distribution = pd.DataFrame({
        "Count": counts,
        "Percentage": (counts / len(df) * 100).round(1)
    })
    print(f"\n{column}")
    print(distribution.to_string())


clinical VQA task
                                   Count  Percentage
clinical VQA task                                   
Disease Diagnosis                    182        60.7
Organ Recognition - Thorax            25         8.3
Bone                                  22         7.3
Severity Grading                      16         5.3
Organ Recognition - Abdomen           14         4.7
Organ Recognition - Head and Neck     13         4.3
Blood Vessels Recognition             10         3.3
Muscle                                 7         2.3
Organ Recognition - Pelvic             7         2.3
Attribute Recognition                  4         1.3

department
                                            Count  Percentage
department                                                   
Pulmonary Medicine                             76        25.3
Sports Medicine                                50        16.7
Orthopedic Surgery                             49        16.3
Gastroenterology and He

## a. Inference with Qwen2-VL-2B-Instruct

In [5]:
from transformers import AutoModelForImageTextToText, AutoProcessor, GenerationConfig
from PIL import Image

model_id = "Qwen/Qwen2-VL-2B-Instruct"

processor = AutoProcessor.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype='auto',
    device_map='auto'
)

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype='auto',
    device_map='auto'
).eval()

preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/56.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

In [6]:
model.device # verify this is running on 'cuda'

device(type='cuda', index=0)

In [7]:
images_dir = "data/images"

def format_inputs(batch_item: Dict[str, Any]) -> Dict[str, Any]:
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": batch_item["prompt"]},
                {"type": "image"},
            ],
        }
    ]

    return {
        **batch_item,
        "messages": messages,
        "image": Image.open(os.path.join(images_dir, batch_item["image_path"]))
    }

In [ ]:
batch_size = 4
max_new_tokens = 32
temperature = 0.6  # <-- Pay close attention to this parameter: you will be asked to explore how changing it affects the results.
top_p = 0.9
sample_answers = 10  # we sample 10 responses for each question.

# Repeat each element in data list 10 times
data = [item for item in data_records for _ in range(sample_answers)]

#if the block takes too long to run, use a subset for initial inspection
#data = data[:100]

results = []
for i in tqdm(range(0, len(data), batch_size)):
    batch = data[i:i + batch_size]
    batch = [format_inputs(item) for item in batch]

    prompts_batch = [processor.apply_chat_template(item["messages"], add_generation_prompt=True) for item in batch]
    images_batch = [item["image"] for item in batch]

    inputs = processor(
        images=images_batch,
        text=prompts_batch,
        return_tensors="pt",
        padding=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False if temperature == 0 else True,
            temperature=temperature,
            top_p=top_p,
            return_dict_in_generate=True,
            output_scores=True,
            output_logits=True,
            pad_token_id=processor.tokenizer.pad_token_id,
        )

    input_ids = inputs["input_ids"]

    for b_idx, item in enumerate(batch):
        input_len = input_ids[b_idx].shape[0]

        generated_tokens = outputs.sequences[b_idx, input_len:]
        generated_text = processor.decode(generated_tokens, skip_special_tokens=True)

        logprobs = []
        for j, logits in enumerate(outputs.logits):  # extract the probability associated to each generated token
            if j >= generated_tokens.shape[0]:  # Avoid out-of-bounds if gen shorter than scores
                break
            token_id = outputs.sequences[b_idx, input_len + j].item()
            probs = torch.nn.functional.softmax(logits[b_idx], dim=-1)
            logprob = torch.log(probs[token_id]).item()
            token_str = processor.tokenizer.decode([token_id])
            logprobs.append((token_str, logprob))

        row = df[df["index"] == item["index"]].iloc[0]

        results.append({
            "index": item["index"],
            "question": item["question"],
            "gold_label": item["answer"],
            "category": item["category"],
            'clinical VQA task': item['clinical VQA task'],
            'department': item['department'],
            'perceptual granularity': item['perceptual granularity'],
            'modality': item['modality'],
            'original task': item['original task'],
            'image_path':item['image_path'],
            "options": row["options"],
            "prompt": prompts_batch[b_idx],
            "response": generated_text,
            "logprobs": logprobs
        })

# save the generated responses
output_df = pd.DataFrame(results)
output_folder = "outputs"
os.makedirs(output_folder, exist_ok=True)

filename = f"outputs_{os.path.split(model_id)[1]}-temp_{temperature}.tsv"
output_df.to_csv(os.path.join(output_folder, filename), index=False, sep="\t", encoding="utf-8")

  0%|          | 2/750 [00:09<57:35,  4.62s/it]


KeyboardInterrupt: 

In [ ]:
# Q2
from pathlib import Path
import ast

# Load files from the folder
folder = Path("/content/data/model_responses")
files = sorted(folder.glob("*.tsv"))

results = pd.concat(
    [pd.read_csv(file, sep="\t") for file in files],
    ignore_index=True
)


# Count each question only once
analysis_df = results.drop_duplicates(subset="index").copy()
n = len(analysis_df)

if n == 0:
    raise ValueError("The files contain no questions.")

print(f"Unique questions: {n}")

# Average question length in words
questions = analysis_df["question"].fillna("").astype(str).str.strip()
word_counts = questions.str.split().str.len()
print(f"\nAverage question length: {word_counts.mean():.2f} words")

# Questions ending with a question mark
ends_with_question_mark = questions.str.endswith("?")
print(
    f"Questions ending with '?': {ends_with_question_mark.sum()} "
    f"({ends_with_question_mark.mean():.1%})"
)


# Number of available answer options per question
def count_options(options):
    if isinstance(options, str):
        options = ast.literal_eval(options)

    if isinstance(options, dict):
        values = list(options.values())
    elif isinstance(options, (list, tuple)):
        values = list(options)

    return sum(
        value is not None
        and pd.notna(value)
        and str(value).strip() != ""
        for value in values
    )

option_counts = analysis_df["options"].apply(count_options)

if option_counts.eq(0).any():
    raise ValueError("Some questions have no available answer options.")

print("\nDistribution of the number of answer options:")
for number, count in option_counts.value_counts().sort_index().items():
    print(f"{number} options: {count} questions ({count / n:.1%})")


# Distribution of gold answer letters.
gold = (
    analysis_df["gold_label"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)

letters = ["A", "B", "C", "D", "E"]
gold_counts = gold.value_counts().reindex(letters, fill_value=0)

print("\nDistribution of gold answer letters:")
for letter, count in gold_counts.items():
    print(f"{letter}: {count} questions ({count / n:.1%})")


# Expected accuracy when choosing uniformly among available options
random_accuracy = (1 / option_counts).mean()

# Accuracy when always choosing the most frequent gold letter
most_frequent_letter = gold_counts.idxmax()
majority_accuracy = gold.eq(most_frequent_letter).mean()

print("\nBaselines:")
print(f"Expected uniform random accuracy: {random_accuracy:.2%}")
print(f"Majority-letter accuracy: {majority_accuracy:.2%}")

Unique questions: 300

Average question length: 14.32 words
Questions ending with '?': 274 (91.3%)

Distribution of the number of answer options:
2 options: 7 questions (2.3%)
3 options: 1 questions (0.3%)
4 options: 210 questions (70.0%)
5 options: 82 questions (27.3%)

Distribution of gold answer letters:
A: 67 questions (22.3%)
B: 75 questions (25.0%)
C: 70 questions (23.3%)
D: 69 questions (23.0%)
E: 19 questions (6.3%)

Baselines:
Expected uniform random accuracy: 24.24%
Majority-letter accuracy: 25.00%


In [ ]:
# Q3
# Load files from the folder
folder = Path("/content/data/model_responses")
files = sorted(folder.glob("*.tsv"))

results = pd.concat(
    [pd.read_csv(file, sep="\t") for file in files],
    ignore_index=True
)

# Select an actual dataset question.
example = results.drop_duplicates(subset="index").iloc[0]

print("Question:")
print(example["question"])

print("\nAnswer options:")
print(example["options"])

print("\nGold answer:")
print(example["gold_label"])

Question:
Which of the following options best matches the marked organ in the X-ray image?

Answer options:
{'A': 'fascia uncovered kidney parenchyma', 'B': 'lung', 'C': 'fascia covered kidney parenchyma', 'D': 'heart', 'E': 'clavicles'}

Gold answer:
D


In [ ]:
#Q4
import pandas as pd
import numpy as np
import ast

# Load responses and parse columns directly
df_raw = pd.read_csv("./data/model_responses/outputs_Qwen2-VL-2B-Instruct-temp_1.0.tsv", sep="\t")
df_raw["options"] = df_raw["options"].apply(ast.literal_eval)
df_raw["parsed_logprobs"] = df_raw["logprobs"].apply(ast.literal_eval)

# Inspect the requested example at index 454
row = df_raw[df_raw["index"] == 454].iloc[0]

print("Question:", row["question"])
print("Options :", row["options"])
print("Gold    :", row["gold_label"])
print("Response (decode output):", repr(row["response"]))
print("\nParsed Logprobs:")
print(f"{'Token':<22}{'Logprob':<10}{'Probability':<12}")
print("-"*44)
for tok, lp in row["parsed_logprobs"]:
    print(f"{tok!r:22}{lp:10.4f}{np.exp(lp):12.6f}")

Question: Which of the following options best matches the marked organ in the X-ray image?
Options : {'A': 'stomach', 'B': 'small intestine', 'C': 'optic cup', 'D': 'kidney boundary', 'E': 'clavicles'}
Gold    : E
Response (decode output): '[D] kidney boundary'

Parsed Logprobs:
Token                 Logprob   Probability 
--------------------------------------------
'[D'                     -0.0345    0.966074
']'                      -0.0033    0.996749
' kidney'                -0.0090    0.991046
' boundary'              -0.0006    0.999385
'<|im_end|>'             -0.0009    0.999118
'<|endoftext|>'         -18.3125    0.000000
'<|endoftext|>'         -13.0555    0.000002
'<|endoftext|>'         -13.1087    0.000002
'<|endoftext|>'          -9.8509    0.000053


## b. Inference with medgemma-4b-it

In [8]:
import gc

# Release the previous model and tensors from its inference loop.
for name in (
    "model", "processor", "inputs", "outputs", "input_ids",
    "generated_tokens", "logits", "probs"
):
    globals().pop(name, None)

gc.collect()
torch.cuda.empty_cache()

In [9]:
from transformers import AutoModelForImageTextToText, AutoProcessor, GenerationConfig
from PIL import Image
import requests

model_id = "google/medgemma-4b-it"

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

processor = AutoProcessor.from_pretrained(model_id)

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [10]:
model.device

device(type='cuda', index=0)

In [11]:
images_dir = "data/images"

def format_inputs(batch_item: Dict[str, Any]) -> Dict[str, Any]:
    image = Image.open(os.path.join(images_dir, batch_item["image_path"]))
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": batch_item["prompt"]},
                {"type": "image", "image":image},
            ],
        }
    ]

    return {
        **batch_item,
        "messages": messages,
        "image": [image],
    }

In [ ]:
batch_size = 1
max_new_tokens = 32
temperature = 0.6  # <-- Pay close attention to this parameter: you will be asked to explore how changing it affects the results.
top_p = 0.9
sample_answers = 10

# Repeat each element in data list 10 times
data = [item for item in data_records for _ in range(sample_answers)]

#if the block takes too long to run, use a subset for initial inspection
#data = data[:100]

results = []
for i in tqdm(range(0, len(data), batch_size)):
    batch = data[i:i + batch_size]
    batch = [format_inputs(item) for item in batch]

    prompts_batch = [processor.apply_chat_template(item["messages"], add_generation_prompt=True) for item in batch]
    images_batch = [item["image"] for item in batch]

    inputs = processor(
        text=prompts_batch,
        images=images_batch,
        padding=True,
        return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False if temperature == 0 else True,
            temperature=temperature,
            top_p=top_p,
            return_dict_in_generate=True,
            output_scores=True,
            output_logits=True,
            pad_token_id=processor.tokenizer.pad_token_id,
        )

    input_ids = inputs["input_ids"]

    for b_idx, item in enumerate(batch):
        input_len = input_ids[b_idx].shape[0]

        generated_tokens = outputs.sequences[b_idx, input_len:]
        generated_text = processor.decode(generated_tokens, skip_special_tokens=True)

        logprobs = []
        for j, logits in enumerate(outputs.scores):
            if j >= generated_tokens.shape[0]:
                break
            token_id = outputs.sequences[b_idx, input_len + j].item()
            probs = torch.nn.functional.softmax(logits[b_idx], dim=-1)
            logprob = torch.log(probs[token_id]).item()
            token_str = processor.tokenizer.decode([token_id])
            logprobs.append((token_str, logprob))

        row = df[df["index"] == item["index"]].iloc[0]

        results.append({
            "index": item["index"],
            "question": item["question"],
            "gold_label": item["answer"],
            "category": item["category"],
            'clinical VQA task': item['clinical VQA task'],
            'department': item['department'],
            'perceptual granularity': item['perceptual granularity'],
            'modality': item['modality'],
            'original task': item['original task'],
            'image_path': item['image_path'],
            "options": row["options"],
            "prompt": prompts_batch[b_idx],
            "response": generated_text,
            "logprobs": logprobs
        })

# save the generated responses
output_df = pd.DataFrame(results)
output_folder = "outputs"
os.makedirs(output_folder, exist_ok=True)

filename = f"outputs_{os.path.split(model_id)[1]}-temp_{temperature}.tsv"
output_df.to_csv(os.path.join(output_folder, filename), index=False, sep="\t", encoding="utf-8")

  0%|          | 3/3000 [00:16<4:34:47,  5.50s/it]


KeyboardInterrupt: 

## Compute and Save Embeddings of Model Outputs
In Section 2, you will compute and evaluate several Uncertainty Estimation (UE) methods. One of these is Radial Dispersion Score (RDS), an embedding-based UE method. To prepare for this, we first need to compute and save the embeddings of all model responses.
We will:

1. Compute and save the embeddings for all model responses.
2. Zip the folder containing the embeddings.
3. Download the zip file.
4. Change the runtime to CPU (see Section 2) and upload the zip file to the runtime.

In [ ]:
# Load the embedding model
from sentence_transformers import SentenceTransformer
import numpy as np
import os

embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
os.makedirs("/content/data/embeddings", exist_ok=True)
embeddings_folder = "/content/data/embeddings"
outputs_folder = "/content/data/model_responses/"

def embed_save_outputs(df, output_path, embedding_model):
    sentences = df["response"].tolist()
    sentences = [str(s) for s in sentences]
    embeddings = embedding_model.encode(sentences, batch_size=32, show_progress_bar=True)
    np.save(output_path, embeddings)

for path in glob.glob(os.path.join(outputs_folder, "*")):
    filename = os.path.basename(path)
    filename = filename.replace("outputs_", "").replace(".tsv", "")
    embedding_file = os.path.join(embeddings_folder, f"embeddings_{filename}.npy")
    df = pd.read_csv(path, sep='\t')
    if not os.path.exists(embedding_file):
        embed_save_outputs(df, embedding_file, embedding_model)

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Batches:   0%|          | 0/65 [00:00<?, ?it/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

In [ ]:
# We will zip the /content/data/embeddings folder, download the zip file manually, since restarting the runtime will delete the results.
! zip -r /content/data/embeddings.zip /content/data/embeddings

  adding: content/data/embeddings/ (stored 0%)
  adding: content/data/embeddings/embeddings_Qwen2-VL-2B-Instruct-temp_1.0.npy (deflated 73%)
  adding: content/data/embeddings/embeddings_medgemma-4b-it-temp_0.6.npy (deflated 86%)
  adding: content/data/embeddings/embeddings_Qwen2-VL-2B-Instruct-temp_0.6.npy (deflated 80%)
  adding: content/data/embeddings/embeddings_gpt5-mini_no-logprobs.npy (deflated 39%)
  adding: content/data/embeddings/embeddings_medgemma-4b-it-temp_1.0.npy (deflated 84%)
  adding: content/data/embeddings/embeddings_Qwen2.5-VL-32B-Instruct-temp_1.0.npy (deflated 23%)
  adding: content/data/embeddings/embeddings_Qwen2.5-VL-32B-Instruct-temp_0.2.npy (deflated 33%)
  adding: content/data/embeddings/embeddings_medgemma-4b-it-temp_0.2.npy (deflated 88%)
  adding: content/data/embeddings/embeddings_Qwen2-VL-2B-Instruct-temp_0.2.npy (deflated 87%)
  adding: content/data/embeddings/embeddings_Qwen2.5-VL-32B-Instruct-temp_0.6.npy (deflated 22%)


# 2. Load and Analyze Outputs (run on CPUs) - Uncertainty Quantification


In [12]:
import re
from collections import Counter
import ast
import numpy as np
import pandas as pd

### a. Utilities and Implementations for
* Answer extraction, accuracy, majority voting
* Uncertainty Estimation (UE) methods:
  * **Token-level probability** (log-likelihood): the probability assigned to each generated token in a response, used as a proxy for model uncertainty. Higher probability = higher confidence. Note: this is not available for `GPT-5-mini`. You have to implement this method (see the assignment report).
  * **Semantic Entropy**: measures the consistency of multiple responses to the same question. More consistent responses across runs = higher model confidence.
  * **Radial Dispersion Score (RDS)**: measures the dispersion of embeddings from their centroid, for multiple responses to the same question. You have to implement this method (see the assignment report). Note: Don't forget to upload the zip file with the saved embeddings, and unzip it!
* Uncertainty Estimation evaluation metrics:
  * Discrimination metric: Area Under the Receiver Operating Characteristic Curve (**AUROC**)
  * Calibration metrics: Expected Calibration Error (**ECE**) and **Brier Score**.


In [13]:
def parse_response(response):
    """Extract the letter choice from the response."""
    # Look for pattern like [A], [B], etc.
    match = re.search(r'\[([A-E])\]', response)
    if match:
        return match.group(1)
    return None

def parse_options(options_str):
    """Parse the options string to get the number of options."""
    try:
        # Convert string representation of dict to actual dict
        options_dict = ast.literal_eval(options_str)
        return len(options_dict)
    except:
        # Fallback: assume 5 options if parsing fails
        return 5

def load_and_process_data(file_path):
    """Load the TSV file and process the data."""
    df = pd.read_csv(file_path, sep='\t')

    # Parse responses to extract letter choices
    df['parsed_response'] = df['response'].apply(parse_response)

    # Parse options to get number of choices per question
    df['num_options'] = df['options'].apply(parse_options)

    return df

def compute_accuracy_stats(df):
    """Compute overall accuracy and per-question statistics."""
    stats = {}

    # Overall accuracy
    correct_predictions = (df['parsed_response'] == df['gold_label']).sum()
    total_predictions = len(df)
    overall_accuracy = correct_predictions / total_predictions

    stats['overall_accuracy'] = overall_accuracy
    stats['total_samples'] = total_predictions
    stats['correct_predictions'] = correct_predictions

    # Per-question statistics (grouped by index, which is unique per question)
    question_stats = []
    unique_indices = df['index'].unique()

    for idx in unique_indices:
        question_data = df[df['index'] == idx]

        # Get all responses for this question (should be 10 samples)
        responses = question_data['parsed_response'].tolist()
        gold_label = question_data['gold_label'].iloc[0]  # Should be same for all samples
        question_text = question_data['question'].iloc[0]  # Should be same for all samples
        num_options = question_data['num_options'].iloc[0]  # Should be same for all samples

        # Calculate accuracy for this question
        correct = sum(1 for r in responses if r == gold_label)
        accuracy = correct / len(responses)

        # Calculate consistency (how often the model gives the same answer)
        response_counts = Counter(responses)
        most_common_response, most_common_count = response_counts.most_common(1)[0]
        consistency = most_common_count / len(responses)

        # Calculate entropy of responses (lower = more consistent)
        response_probs = np.array(list(response_counts.values())) / len(responses)
        entropy = -np.sum(response_probs * np.log2(response_probs))

        question_stats.append({
            'index': idx,
            'question': question_text[:50] + '...' if len(question_text) > 50 else question_text,
            'gold_label': gold_label,
            'accuracy': accuracy,
            'consistency': consistency,
            'most_common_response': most_common_response,
            'entropy': entropy,
            'num_samples': len(responses),
            'num_options': num_options
        })

    stats['question_stats'] = question_stats
    return stats


In [14]:
def compute_semantic_entropy_uncertainty(df):
    """
    Compute semantic entropy as uncertainty metric and evaluate it.
    For each question, semantic entropy measures how uncertain the model is.
    """
    grouped = df.groupby('index')

    uncertainties = []
    correctness = []  # Binary: 1 if majority answer is correct, 0 otherwise
    num_options_list = []  # Track number of options for each question

    for idx, group in grouped:
        responses = group['parsed_response'].tolist()
        gold_label = group['gold_label'].iloc[0]
        num_options = group['num_options'].iloc[0]

        # Count responses
        response_counts = Counter(responses)
        total_responses = len(responses)

        # Calculate semantic entropy (uncertainty)
        response_probs = np.array(list(response_counts.values())) / total_responses
        semantic_entropy = -np.sum(response_probs * np.log2(response_probs))

        # Determine if majority answer is correct
        most_common_response = response_counts.most_common(1)[0][0]
        is_correct = 1 if most_common_response == gold_label else 0

        uncertainties.append(semantic_entropy)
        correctness.append(is_correct)
        num_options_list.append(num_options)

    return np.array(uncertainties), np.array(correctness), np.array(num_options_list)


def compute_calibration_metrics(uncertainties, correctness, n_bins=10):
    """Compute AUROC, ECE, and Brier score with dynamic normalization."""

    from sklearn.metrics import roc_auc_score

    # AUROC: How well does uncertainty predict incorrectness?
    # Higher uncertainty should predict lower correctness
    try:
        auroc = roc_auc_score(1 - correctness,
                              uncertainties)  # 1-correctness because higher uncertainty should predict incorrectness
    except ValueError:
        auroc = 0.5  # If all samples have same correctness

    # Expected Calibration Error (ECE)
    def compute_ece(uncertainties, correctness, n_bins=10):
        bin_edges = np.linspace(0, np.max(uncertainties) + 1e-8, n_bins + 1)
        ece = 0

        for i in range(n_bins):
            mask = (uncertainties >= bin_edges[i]) & (uncertainties < bin_edges[i + 1])
            if np.sum(mask) > 0:
                bin_accuracy = np.mean(correctness[mask])
                bin_confidence = 1 - np.mean(uncertainties[mask])
                bin_confidence = np.clip(bin_confidence, 0, 1)  # Ensure it's between 0 and 1
                bin_size = np.sum(mask) / len(uncertainties)
                ece += bin_size * abs(bin_accuracy - bin_confidence)

        return ece

    u_min, u_max = uncertainties.min(), uncertainties.max()
    if u_max > u_min:
        uncertainties = (uncertainties - u_min) / (u_max - u_min)
    else:
        uncertainties = np.zeros_like(uncertainties)
    confidences = 1 - uncertainties
    confidences = np.clip(confidences, 0, 1)

    ece = compute_ece(uncertainties, correctness, n_bins)
    brier_score = np.mean((confidences - correctness) ** 2)

    return auroc, ece, brier_score

### b. Run Evaluation

In [15]:
# Replace this with the output file corresponding to the model and decoding configuration you want to evaluate.
file_path = "./data/model_responses/outputs_Qwen2-VL-2B-Instruct-temp_1.0.tsv"

print("Loading and processing data...")
df = load_and_process_data(file_path)

print(f"Loaded {len(df)} samples")
print(f"Number of unique questions (indices): {df['index'].nunique()}")
print(f"Samples per question: {len(df) // df['index'].nunique()}")

# Print option distribution
option_counts = df.groupby('index')['num_options'].first().value_counts().sort_index()
print(f"Distribution of number of options:")
for num_opts, count in option_counts.items():
    print(f"  {num_opts} options: {count} questions")

Loading and processing data...
Loaded 3000 samples
Number of unique questions (indices): 300
Samples per question: 10
Distribution of number of options:
  2 options: 7 questions
  3 options: 1 questions
  4 options: 210 questions
  5 options: 82 questions


In [ ]:
# Basic accuracy statistics
print("\n" + "=" * 50)
print("ACCURACY ANALYSIS")
print("=" * 50)

stats = compute_accuracy_stats(df)
print(f"Overall Accuracy: {stats['overall_accuracy']:.3f}")
print(f"Correct Predictions: {stats['correct_predictions']}/{stats['total_samples']}")

# Per-question statistics
print(f"\nPer-Question Statistics:")
print("-" * 100)
print(f"{'Index':<6} {'Question':<50} {'Gold':<5} {'Opts':<7} {'Acc':<6} {'Cons':<2} {'Pred':<4} {'Entropy':<7}")
print("-" * 100)
for q_stat in stats['question_stats']:
    print(f"{q_stat['index']:<6} {q_stat['question']:<40} {q_stat['gold_label']:<4} "
          f"{q_stat['num_options']:<4} {q_stat['accuracy']:<6.3f} {q_stat['consistency']:<6.3f} "
          f"{q_stat['most_common_response']:<4} {q_stat['entropy']:<7.3f}")


ACCURACY ANALYSIS
Overall Accuracy: 0.316
Correct Predictions: 948/3000

Per-Question Statistics:
----------------------------------------------------------------------------------------------------
Index  Question                                           Gold  Opts    Acc    Cons Pred Entropy
----------------------------------------------------------------------------------------------------
398    Which of the following options best matches the ma... D    5    0.100  0.600  B    1.295  
4322   Observe the image, please select the option that b... A    4    0.000  0.900  C    0.469  
2627   Considering the symptoms visible in the picture, w... B    3    0.800  0.800  B    0.722  
70     Focus on the square-highlighted area of this X-ray... A    4    0.200  0.700  B    1.157  
4525   Choose the description that best matches the image... C    4    0.000  0.600  D    0.971  
2277   Which of the following options best matches the co... C    4    0.000  0.500  D    1.361  
57     Observe

In [ ]:
# Uncertainty quantification analysis
print("\n" + "=" * 60)
print("UNCERTAINTY QUANTIFICATION ANALYSIS - Semantic Entropy")
print("=" * 60)

uncertainties, correctness, num_options_array = compute_semantic_entropy_uncertainty(df)
auroc, ece, brier_score = compute_calibration_metrics(uncertainties, correctness)

print(f"\n Discrimination and Calibration Metrics:\n")
print(f" * AUROC (uncertainty vs incorrectness): {auroc:.3f}")
print(f" * Expected Calibration Error (ECE): {ece:.3f}")
print(f" * Brier Score: {brier_score:.3f}")


UNCERTAINTY QUANTIFICATION ANALYSIS - Semantic Entropy

 Discrimination and Calibration Metrics:

 * AUROC (uncertainty vs incorrectness): 0.595
 * Expected Calibration Error (ECE): 0.159
 * Brier Score: 0.256


In [ ]:
# Q5
example = df[df["index"] == 767]

print("Example question:")
print(example["question"].iloc[0])
print("Number of responses:", len(example))
print("\nAnswer counts:")
print(example["parsed_response"].value_counts(dropna=False))

counts = example["parsed_response"].value_counts(dropna=False)
probs = counts.to_numpy() / counts.sum()

manual_entropy = -np.sum(probs * np.log2(probs))

print("\nManual entropy:", round(manual_entropy, 3))


Example question:
This is a X-ray image. Which of the following options is the most appropriate to demonstrate the marked area?
Number of responses: 10

Answer counts:
parsed_response
E    5
B    3
A    1
C    1
Name: count, dtype: int64

Manual entropy: 1.685


In [ ]:
# Q6
example = df[df['index'] == 755]

print("Question:", example['question'].iloc[0])
print("Options:", example['options'].iloc[0])
print("Gold answer:", example['gold_label'].iloc[0])
print("\nSampled responses:\n")

for i, (_, row) in enumerate(example.iterrows(), start=1):
    print(f"Sample {i}:")
    print("Full response:", row['response'])
    print("Extracted answer:", row['parsed_response'])
    print("-" * 50)

Question: Which of the following options best matches the symptoms of marked region shown in the X-ray image?
Options: {'A': 'cardiomegaly', 'B': 'lung nodule', 'C': 'pneumothorax', 'D': 'lung calcification', 'E': 'atelectasis'}
Gold answer: E

Sampled responses:

Sample 1:
Full response: [D]
Extracted answer: D
--------------------------------------------------
Sample 2:
Full response: [B] lung nodule
Extracted answer: B
--------------------------------------------------
Sample 3:
Full response: [A] cardiomegaly
Extracted answer: A
--------------------------------------------------
Sample 4:
Full response: [B]
Extracted answer: B
--------------------------------------------------
Sample 5:
Full response: [D] lung calcification
Extracted answer: D
--------------------------------------------------
Sample 6:
Full response: [C] pneumothorax
Extracted answer: C
--------------------------------------------------
Sample 7:
Full response: [B]
Extracted answer: B
-----------------------------

In [ ]:
# Q7
files = sorted(glob.glob("./data/model_responses/*.tsv"))

results = []

for file_path in files:
    # Load the saved responses
    df_config = load_and_process_data(file_path)

    # Calculate entropy and correctness
    uncertainties, correctness, _ = compute_semantic_entropy_uncertainty(df_config)

    # Calculate evaluation metrics
    auroc, ece, brier = compute_calibration_metrics(
        uncertainties, correctness
    )

    results.append({
        "Configuration": file_path.split("/")[-1].replace("outputs_", "").replace(".tsv", ""),
        "AUROC": auroc,
        "ECE": ece,
        "Brier Score": brier
    })

results_df = pd.DataFrame(results)

display(results_df.round(3))
print("Best AUROC (highest):")
display(results_df.loc[[results_df["AUROC"].idxmax()]].round(3))

print("\nBest ECE (lowest):")
display(results_df.loc[[results_df["ECE"].idxmin()]].round(3))

print("\nBest Brier Score (lowest):")
display(results_df.loc[[results_df["Brier Score"].idxmin()]].round(3))

,Configuration,AUROC,ECE,Brier Score
0,Qwen2-VL-2B-Instruct-temp_0.2,0.530,0.539,0.544
1,Qwen2-VL-2B-Instruct-temp_0.6,0.578,0.322,0.362
2,Qwen2-VL-2B-Instruct-temp_1.0,0.595,0.159,0.256
3,Qwen2.5-VL-32B-Instruct-temp_0.2,0.549,0.419,0.425
4,Qwen2.5-VL-32B-Instruct-temp_0.6,0.608,0.293,0.352
5,Qwen2.5-VL-32B-Instruct-temp_1.0,0.670,0.211,0.278
6,gpt5-mini_no-logprobs,0.638,0.226,0.238
7,medgemma-4b-it-temp_0.2,0.518,0.587,0.585
8,medgemma-4b-it-temp_0.6,0.532,0.540,0.545
9,medgemma-4b-it-temp_1.0,0.552,0.519,0.521


Best AUROC (highest):


,Configuration,AUROC,ECE,Brier Score
5,Qwen2.5-VL-32B-Instruct-temp_1.0,0.67,0.211,0.278



Best ECE (lowest):


,Configuration,AUROC,ECE,Brier Score
2,Qwen2-VL-2B-Instruct-temp_1.0,0.595,0.159,0.256



Best Brier Score (lowest):


,Configuration,AUROC,ECE,Brier Score
6,gpt5-mini_no-logprobs,0.638,0.226,0.238


In [ ]:
#Q8
import pandas as pd
import re
from collections import Counter

files = {
    "0.2": "./data/model_responses/outputs_Qwen2.5-VL-32B-Instruct-temp_0.2.tsv",
    "0.6": "./data/model_responses/outputs_Qwen2.5-VL-32B-Instruct-temp_0.6.tsv",
    "1.0": "./data/model_responses/outputs_Qwen2.5-VL-32B-Instruct-temp_1.0.tsv"
}

for temp, path in files.items():
    df = pd.read_csv(path, sep="\t")

    # Extract answer letters A-E; invalid responses become None
    df["answer"] = df["response"].apply(
        lambda x: re.search(r"\[([A-E])\]", x).group(1)
        if isinstance(x, str) and re.search(r"\[([A-E])\]", x)
        else None
    )

    # Accuracy across individual responses (invalid = incorrect)
    sample_acc = (df["answer"] == df["gold_label"]).mean()

    correct_votes = 0
    fractions = []

    for idx, group in df.groupby("index"):
        gold = group["gold_label"].iloc[0]
        answers = group["answer"].dropna().tolist()

        # Fraction of correct sampled responses
        fractions.append(
            (group["answer"] == gold).mean()
        )

        if answers:
            counts = Counter(answers)
            highest = max(counts.values())
            # Alphabetical tie-break
            majority = min(a for a in counts if counts[a] == highest)
        else:
            majority = None

        if majority == gold:
            correct_votes += 1

    majority_acc = correct_votes / df["index"].nunique()

    print(f"Temperature {temp}:")
    print(f"  Individual accuracy: {sample_acc:.2%}")
    print(f"  Majority-vote accuracy: {majority_acc:.2%}")

    # Find an example where some responses are correct,
    # but the majority-vote answer is incorrect
    if temp == "1.0":
        for idx, group in df.groupby("index"):
            gold = group["gold_label"].iloc[0]
            answers = group["answer"].dropna().tolist()
            fraction = (group["answer"] == gold).mean()

            if answers and 0 < fraction < 1:
                counts = Counter(answers)
                highest = max(counts.values())
                majority = min(a for a in counts if counts[a] == highest)

                if majority != gold:
                    print("\nExample question:", idx)
                    print("Question:", group["question"].iloc[0])
                    print("Gold answer:", gold)
                    print("Majority answer:", majority)
                    print(f"Fraction correct: {fraction:.1%}")
                    break


Temperature 0.2:
  Individual accuracy: 51.40%
  Majority-vote accuracy: 51.33%
Temperature 0.6:
  Individual accuracy: 49.73%
  Majority-vote accuracy: 52.00%
Temperature 1.0:
  Individual accuracy: 49.63%
  Majority-vote accuracy: 53.00%

Example question: 5
Question: Looking at the area highlighted by a box in this X-ray image, which of the following conditions is most consistent with the finding?
Gold answer: B
Majority answer: D
Fraction correct: 10.0%


## Q10

In [ ]:

# Q10 - Extract label-prefix confidence and inspect tokenisation

file_paths = [
    "./data/model_responses/outputs_Qwen2-VL-2B-Instruct-temp_1.0.tsv",
    "./data/model_responses/outputs_Qwen2.5-VL-32B-Instruct-temp_0.2.tsv",
    "./data/model_responses/outputs_Qwen2.5-VL-32B-Instruct-temp_0.6.tsv",
    "./data/model_responses/outputs_Qwen2.5-VL-32B-Instruct-temp_1.0.tsv"
]

def get_label_confidence(logprobs):
    for i, (token, logprob) in enumerate(logprobs):

        # Prefix in one token, e.g. '[B'
        if re.fullmatch(r'\[[A-E]', token):
            return np.exp(logprob)

        # Prefix split into two tokens, e.g. '[' and 'B'
        if token.strip() == '[' and i + 1 < len(logprobs):
            next_token, next_logprob = logprobs[i + 1]

            if re.fullmatch(r'[A-E]', next_token.strip()):
                return np.exp(logprob + next_logprob)

    return np.nan


def get_tokenization_type(logprobs):
    for i, (token, logprob) in enumerate(logprobs):

        if re.fullmatch(r'\[[A-E]', token):
            return "One token"

        if token.strip() == '[' and i + 1 < len(logprobs):
            next_token = logprobs[i + 1][0]

            if re.fullmatch(r'[A-E]', next_token.strip()):
                return "Two tokens"

    return "Other / not detected"


# Exploratory analysis of Qwen2-VL-2B at temperature 1.0
df_raw = load_and_process_data(file_paths[0])

df_raw["parsed_logprobs"] = df_raw["logprobs"].apply(ast.literal_eval)

df_raw["label_confidence"] = df_raw["parsed_logprobs"].apply(
    get_label_confidence
)

df_raw["tokenization_type"] = df_raw["parsed_logprobs"].apply(
    get_tokenization_type
)

print("Tokenisation patterns:")
print(df_raw["tokenization_type"].value_counts())

print("\nResponses without extracted confidence:")
display(
    df_raw.loc[
        df_raw["label_confidence"].isna(),
        ["response", "logprobs"]
    ]
)


Tokenisation patterns:
tokenization_type
One token               2954
Two tokens                32
Other / not detected      14
Name: count, dtype: int64

Responses without extracted confidence:


,response,logprobs
124,[2],"[('[', -2.8539485931396484), ('2', -5.04449272..."
202,[AB],"[('[', -3.363659620285034), ('AB', -3.09508895..."
362,[AB],"[('[', -3.037727117538452), ('AB', -3.44784855..."
800,[BC],"[('[', -3.397300958633423), ('BC', -4.32741928..."
995,[BCD],"[('[', -3.4789950847625732), ('BC', -2.3868968..."
1126,E,"[('E', -3.413950204849243), ('<|im_end|>', -0...."
1824,[0],"[('[', -3.3052117824554443), ('0', -2.63000226..."
1915,[Ch],"[('[', -3.419769287109375), ('Ch', -3.62270855..."
1999,[CD],"[('[', -3.404648780822754), ('CD', -3.56488704..."
2271,E,"[('E', -3.2421393394470215), ('<|im_end|>', -0..."


In [ ]:
# Q10 - Majority-vote confidence and evaluation

from sklearn.metrics import roc_auc_score
from collections import Counter

def get_question_confidence(df):
    results = []

    for idx, group in df.groupby("index"):

        labels = group["parsed_response"].dropna()

        if labels.empty:
            continue

        majority_label = Counter(labels).most_common(1)[0][0]

        # Select responses supporting the majority answer
        matching = group[
            group["parsed_response"] == majority_label
        ]

        results.append({
            "index": idx,
            "majority_label": majority_label,
            "confidence": matching["label_confidence"].mean(),
            "correct": int(
                majority_label == str(group["gold_label"].iloc[0]).strip()
            )
        })

    return pd.DataFrame(results)


def evaluate_confidence(results_df):
    results_df = results_df.dropna(subset=["confidence"])

    confidence = results_df["confidence"].to_numpy()
    correctness = results_df["correct"].to_numpy()

    auroc = roc_auc_score(1 - correctness, 1 - confidence)
    brier = np.mean((confidence - correctness) ** 2)

    ece = 0
    bins = np.linspace(0, 1, 11)

    for i in range(10):
        if i == 9:
            mask = (confidence >= bins[i]) & (confidence <= bins[i + 1])
        else:
            mask = (confidence >= bins[i]) & (confidence < bins[i + 1])

        if mask.sum() > 0:
            ece += mask.mean() * abs(
                correctness[mask].mean() - confidence[mask].mean()
            )

    return {
        "AUROC": round(auroc, 3),
        "ECE": round(ece, 3),
        "Brier": round(brier, 3),
        "Questions": len(results_df)
    }


# Evaluate all four configurations
results_all = []

for file in file_paths:

    df = load_and_process_data(file)

    df["parsed_logprobs"] = df["logprobs"].apply(ast.literal_eval)

    df["label_confidence"] = df["parsed_logprobs"].apply(
        get_label_confidence
    )

    question_results = get_question_confidence(df)
    metrics = evaluate_confidence(question_results)

    metrics["Model"] = (
        "Qwen2.5-VL-32B" if "32B" in file else "Qwen2-VL-2B"
    )
    metrics["Temperature"] = file.split("temp_")[-1].replace(".tsv", "")

    results_all.append(metrics)

print("Evaluation results:")
display(pd.DataFrame(results_all))


# Two examples

print("\nHighest label-prefix confidence:")
display(
    df_raw.nlargest(1, "label_confidence")[
        ["index", "response", "gold_label", "label_confidence"]
    ]
)

print("\nLowest label-prefix confidence:")
display(
    df_raw.nsmallest(1, "label_confidence")[
        ["index", "response", "gold_label", "label_confidence"]
    ]
)

Evaluation results:


,AUROC,ECE,Brier,Questions,Model,Temperature
0,0.546,0.197,0.276,300,Qwen2-VL-2B,1.0
1,0.678,0.218,0.279,300,Qwen2.5-VL-32B,0.2
2,0.699,0.221,0.276,300,Qwen2.5-VL-32B,0.6
3,0.691,0.214,0.276,300,Qwen2.5-VL-32B,1.0



Highest label-prefix confidence:


,index,response,gold_label,label_confidence
600,454,[D] kidney boundary,E,0.966074



Lowest label-prefix confidence:


,index,response,gold_label,label_confidence
1115,1572,[A],D,0.000518


## Q11

In [ ]:
import torch
import torch.nn.functional as F

def compute_radial_dispersion_score(embeddings):
    # L2-normalise each embedding
    u = F.normalize(embeddings, p=2, dim=-1)

    # Calculate the mean embedding (centroid)
    u_bar = torch.mean(u, dim=0, keepdim=True)

    # Sum L1 distances from the centroid
    rds = torch.sum(torch.abs(u - u_bar))

    return rds.item()

In [ ]:
from sklearn.metrics import roc_auc_score, brier_score_loss
%pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer


embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
# Files for the four model configurations
file_paths = {
    "Qwen2-VL-2B (T=1.0)": "./data/model_responses/outputs_Qwen2-VL-2B-Instruct-temp_1.0.tsv",
    "Qwen2.5-VL-32B (T=0.2)": "./data/model_responses/outputs_Qwen2.5-VL-32B-Instruct-temp_0.2.tsv",
    "Qwen2.5-VL-32B (T=0.6)": "./data/model_responses/outputs_Qwen2.5-VL-32B-Instruct-temp_0.6.tsv",
    "Qwen2.5-VL-32B (T=1.0)": "./data/model_responses/outputs_Qwen2.5-VL-32B-Instruct-temp_1.0.tsv"
}

# Expected Calibration Error (ECE)
def calculate_ece(correct, confidence, n_bins=10):
    ece = 0.0
    bin_edges = np.linspace(0, 1, n_bins + 1)

    for i in range(n_bins):
        if i == n_bins - 1:
            mask = (confidence >= bin_edges[i]) & (confidence <= bin_edges[i + 1])
        else:
            mask = (confidence >= bin_edges[i]) & (confidence < bin_edges[i + 1])

        if mask.any():
            bin_accuracy = correct[mask].mean()
            bin_confidence = confidence[mask].mean()
            ece += mask.mean() * abs(bin_accuracy - bin_confidence)

    return ece


# Evaluate RDS for each model
all_results = []

for model_name, path in file_paths.items():

    # Use the provided data-loading function
    data = load_and_process_data(path)

    question_scores = []
    question_correct = []

    for question_id, group in data.groupby("index"):

        # Get the ten generated responses
        responses = group["response"].fillna("").astype(str).tolist()

        # Generate response embeddings
        embeddings = embedding_model.encode(
            responses,
            convert_to_tensor=True
        )

        # Calculate Radial Dispersion Score
        rds = compute_radial_dispersion_score(embeddings)

        # Majority-vote prediction using provided parsed responses
        answers = group["parsed_response"].tolist()
        prediction = Counter(answers).most_common(1)[0][0]
        gold = str(group["gold_label"].iloc[0]).strip()

        # Store uncertainty and correctness
        question_scores.append(rds)
        question_correct.append(int(prediction == gold))

    rds_scores = np.array(question_scores)
    correct = np.array(question_correct)

    # Convert RDS into confidence using the provided normalization approach
    u_min, u_max = rds_scores.min(), rds_scores.max()

    if u_max > u_min:
        normalized_rds = (rds_scores - u_min) / (u_max - u_min)
    else:
        normalized_rds = np.zeros_like(rds_scores)

    confidence = 1 - normalized_rds

    # Evaluation metrics
    auroc = roc_auc_score(1 - correct, rds_scores)
    ece = calculate_ece(correct, confidence)
    brier = brier_score_loss(correct, confidence)

    all_results.append({
        "Model": model_name,
        "Questions": len(correct),
        "AUROC": auroc,
        "ECE": ece,
        "Brier": brier
    })

# Display results
results_df = pd.DataFrame(all_results)
display(results_df.round(3))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,Model,Questions,AUROC,ECE,Brier
0,Qwen2-VL-2B (T=1.0),300,0.542,0.168,0.259
1,Qwen2.5-VL-32B (T=0.2),300,0.509,0.167,0.297
2,Qwen2.5-VL-32B (T=0.6),300,0.581,0.123,0.262
3,Qwen2.5-VL-32B (T=1.0),300,0.642,0.132,0.252


In [ ]:
#Q12
from sklearn.metrics import roc_auc_score

# One row per question: modality, semantic entropy
df = load_and_process_data("./data/model_responses/outputs_Qwen2.5-VL-32B-Instruct-temp_1.0.tsv")
unc, correct, _ = compute_semantic_entropy_uncertainty(df)   # same question order as groupby("index")
q = pd.DataFrame({"modality": df.groupby("index")["modality"].first().values, "unc": unc, "correct": correct})

# Normalise once over ALL questions, so confidences are comparable across modalities --> had to normalize otherwise calculating this was not possible
q["conf"] = 1 - (q["unc"] - q["unc"].min()) / (q["unc"].max() - q["unc"].min())

def metrics(g):
    both_classes = g["correct"].nunique() == 2
    return pd.Series({
        "Questions": len(g),
        "Accuracy": g["correct"].mean(),
        "AUROC": roc_auc_score(1 - g["correct"], g["unc"]) if both_classes else np.nan,
        "ECE": calculate_ece(g["correct"].values, g["conf"].values),
        "Brier": np.mean((g["conf"] - g["correct"]) ** 2),
    })

table = q.groupby("modality").apply(metrics)
table.loc["ALL"] = metrics(q)
display(table.sort_values("Accuracy", ascending=False).round(3))

/tmp/ipykernel_837/727121877.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  table = q.groupby("modality").apply(metrics)


,Questions,Accuracy,AUROC,ECE,Brier
modality,,,,,
MRI,100.0,0.620,0.753,0.168,0.210
ALL,300.0,0.517,0.670,0.211,0.278
CT,100.0,0.480,0.635,0.289,0.312
X-ray,100.0,0.450,0.611,0.245,0.312


In [18]:
#Q13
files = {
    "GPT-5-mini": "./data/model_responses/outputs_gpt5-mini_no-logprobs.tsv",
    "Qwen2.5-VL-32B (T=0.6)": "./data/model_responses/outputs_Qwen2.5-VL-32B-Instruct-temp_0.6.tsv",
}
dfs = {name: load_and_process_data(path) for name, path in files.items()}

# Checking that both models were evaluated on the same questions (with the same number of samples)
gpt, qwen = dfs["GPT-5-mini"], dfs["Qwen2.5-VL-32B (T=0.6)"]
common = set(gpt["index"]) & set(qwen["index"])
print("Common questions:", len(common))

rows = []
for name, d in dfs.items():
    d = d[d["index"].isin(common)]
    unc, correct, _ = compute_semantic_entropy_uncertainty(d)
    auroc, ece, brier = compute_calibration_metrics(unc, correct)
    rows.append({"Model": name, "Questions": len(correct), "Majority-vote accuracy": correct.mean(),
                 "AUROC": auroc, "ECE": ece, "Brier": brier})

display(pd.DataFrame(rows).round(3))

Common questions: 225


,Model,Questions,Majority-vote accuracy,AUROC,ECE,Brier
0,GPT-5-mini,225,0.693,0.638,0.226,0.238
1,Qwen2.5-VL-32B (T=0.6),225,0.560,0.657,0.252,0.298
